<a href="https://colab.research.google.com/github/charvie-web/flyrank-a9-polite-scraper/blob/main/FlyRank_A9_Polite_Scraper.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:

!pip install requests beautifulsoup4 pydantic


In [ ]:

import requests

url = "https://books.toscrape.com/robots.txt"

headers = {
    "User-Agent": "FlyRankInternship-A9/1.0 (educational scraper)"
}

try:
    response = requests.get(
        url,
        headers=headers,
        timeout=10
    )

    print("HTTP Status:", response.status_code)

    if response.status_code == 200:
        print("Robots.txt content:")
        print(response.text)
    elif response.status_code == 404:
        print("No robots file found")
    else:
        print("Robots check returned:", response.status_code)

except requests.RequestException as error:
    print("Request failed:", error)


HTTP Status: 404
No robots file found


In [ ]:

import requests
import os
import time

# Website
URL = "https://books.toscrape.com/"

# Cache folder
os.makedirs("cache", exist_ok=True)

CACHE_FILE = "cache/catalogue-page-1.html"

# Identify our scraper
HEADERS = {
    "User-Agent": "FlyRankInternship-A9/1.0 (educational scraper)"
}

def fetch_page(url, cache_file):

    # Reuse saved HTML if available
    if os.path.exists(cache_file):
        with open(cache_file, "r", encoding="utf-8") as file:
            html = file.read()

        print("CACHE HIT")
        print("Response size:", len(html), "characters")
        return html

    try:
        response = requests.get(
            url,
            headers=HEADERS,
            timeout=10
        )

        print("HTTP Status:", response.status_code)

        if response.status_code != 200:
            print("Fetch failed")
            return None

        html = response.text

        # Save HTML to cache
        with open(cache_file, "w", encoding="utf-8") as file:
            file.write(html)

        print("FETCH")
        print("Response size:", len(html), "characters")
        print("HTML saved successfully!")

        return html

    except requests.RequestException as error:
        print("Request failed:", error)
        return None


# First fetch
html = fetch_page(URL, CACHE_FILE)


HTTP Status: 200
FETCH
Response size: 51294 characters
HTML saved successfully!


In [ ]:
html = fetch_page(URL, CACHE_FILE)

CACHE HIT
Response size: 51294 characters


In [ ]:

from bs4 import BeautifulSoup
from urllib.parse import urljoin
import time

BASE_URL = "https://books.toscrape.com/"

catalogue_url = BASE_URL
all_book_urls = []
catalogue_pages = []

while catalogue_url and len(catalogue_pages) < 3:

    # Use cache for page 1
    if catalogue_url == BASE_URL:
        html = fetch_page(URL, CACHE_FILE)

    else:
        time.sleep(0.5)
        html = requests.get(
            catalogue_url,
            headers=HEADERS,
            timeout=10
        ).text

    soup = BeautifulSoup(html, "html.parser")

    catalogue_pages.append(catalogue_url)

    # Find book links
    for book in soup.select("article.product_pod h3 a"):
        book_url = urljoin(catalogue_url, book["href"])
        all_book_urls.append(book_url)

    # Follow the website's Next link
    next_link = soup.select_one("li.next a")

    if next_link:
        catalogue_url = urljoin(
            catalogue_url,
            next_link["href"]
        )
    else:
        catalogue_url = None

# Remove duplicates
all_book_urls = list(dict.fromkeys(all_book_urls))

print("Catalogue pages:", len(catalogue_pages))
print("Discovered:", len(all_book_urls))
print("Unique URLs:", len(set(all_book_urls)))

print("\nFirst 3 book URLs:")
for book in all_book_urls[:3]:
    print(book)


CACHE HIT
Response size: 51294 characters
Catalogue pages: 3
Discovered: 60
Unique URLs: 60

First 3 book URLs:
https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html
https://books.toscrape.com/catalogue/tipping-the-velvet_999/index.html
https://books.toscrape.com/catalogue/soumission_998/index.html


In [ ]:

import json
import os
import time
from datetime import datetime, timezone
from bs4 import BeautifulSoup

os.makedirs("cache/books", exist_ok=True)

raw_records = []
failed_urls = []
last_request_time = 0

def fetch_book(url):
    global last_request_time

    filename = str(abs(hash(url))) + ".html"
    cache_path = os.path.join("cache/books", filename)

    if os.path.exists(cache_path):
        with open(cache_path, "r", encoding="utf-8") as f:
            return f.read(), "CACHE", None

    # Wait at least 0.5 seconds between real requests
    wait = 0.5 - (time.monotonic() - last_request_time)
    if wait > 0:
        time.sleep(wait)

    try:
        response = requests.get(
            url,
            headers=HEADERS,
            timeout=10
        )

        last_request_time = time.monotonic()

        if response.status_code != 200:
            return None, "FAILED", response.status_code

        html = response.text

        with open(cache_path, "w", encoding="utf-8") as f:
            f.write(html)

        return html, "FETCH", None

    except requests.RequestException as error:
        return None, "FAILED", str(error)


for index, book_url in enumerate(all_book_urls):

    try:
        html, status, error = fetch_book(book_url)

        if html is None:
            failed_urls.append({
                "url": book_url,
                "reason": str(error)
            })
            continue

        soup = BeautifulSoup(html, "html.parser")

        title = soup.select_one("div.product_main h1")
        price = soup.select_one("p.price_color")
        availability = soup.select_one("p.availability")
        rating = soup.select_one("p.star-rating")

        description_heading = soup.select_one(
            "#product_description"
        )

        description = None

        if description_heading:
            description_paragraph = (
                description_heading.find_next_sibling("p")
            )
            if description_paragraph:
                description = description_paragraph.get_text(
                    " ", strip=True
                )

        if not all([title, price, availability, rating]):
            failed_urls.append({
                "url": book_url,
                "reason": "Missing required book fields"
            })
            continue

        rating_classes = rating.get("class", [])
        rating_text = next(
            (word for word in rating_classes
             if word != "star-rating"),
            None
        )

        source_index = index // 20
        source_page = catalogue_pages[source_index]

        record = {
            "title": title.get_text(strip=True),
            "product_url": book_url,
            "price_text": price.get_text(strip=True),
            "availability_text": availability.get_text(" ", strip=True),
            "rating_text": rating_text,
            "description": description,
            "source_page": source_page,
            "fetched_at": datetime.now(timezone.utc).isoformat()
        }

        raw_records.append(record)

    except Exception as error:
        failed_urls.append({
            "url": book_url,
            "reason": str(error)
        })

    if (index + 1) % 10 == 0:
        print("Processed:", index + 1, "/ 60")

# Save raw records
with open("raw_books.json", "w", encoding="utf-8") as f:
    json.dump(raw_records, f, indent=2, ensure_ascii=False)

print("\nRAW EXTRACTION SUMMARY")
print("Successful records:", len(raw_records))
print("Failed pages:", len(failed_urls))
print("Saved: raw_books.json")


Processed: 10 / 60
Processed: 20 / 60
Processed: 30 / 60
Processed: 40 / 60
Processed: 50 / 60
Processed: 60 / 60

RAW EXTRACTION SUMMARY
Successful records: 60
Failed pages: 0
Saved: raw_books.json


In [ ]:

from pydantic import BaseModel, Field, field_validator
from typing import Optional
import re
import json
import os

# Create output directory
os.makedirs("output", exist_ok=True)

# Define our schema
class Book(BaseModel):
    title: str
    product_url: str
    price_text: str
    price_gbp: float = Field(ge=0)
    availability_text: str
    rating_text: str
    description: Optional[str] = None
    source_page: str
    fetched_at: str

    @field_validator(
        "product_url", "source_page"
    )
    @classmethod
    def check_https(cls, value):
        if not value.startswith("https://"):
            raise ValueError("URL must start with https://")
        return value

    @field_validator("title", "price_text", "rating_text")
    @classmethod
    def check_not_empty(cls, value):
        if not value.strip():
            raise ValueError("Field cannot be empty")
        return value


valid_books = []
invalid_books = []
seen_urls = set()

for record in raw_records:
    try:
        # Remove currency symbol and convert to float
        price = float(
            re.sub(r"[^0-9.]", "", record["price_text"])
        )

        clean_record = {
            **record,
            "price_gbp": price
        }

        # Remove duplicate books
        if clean_record["product_url"] in seen_urls:
            continue

        # Validate with Pydantic
        book = Book.model_validate(clean_record)

        valid_books.append(
            book.model_dump()
        )

        seen_urls.add(clean_record["product_url"])

    except Exception as error:
        invalid_books.append({
            "record": record,
            "reason": str(error)
        })

# Save valid records
with open("output/books.json", "w", encoding="utf-8") as f:
    json.dump(valid_books, f, indent=2, ensure_ascii=False)

# Save invalid records
with open("output/errors.json", "w", encoding="utf-8") as f:
    json.dump(invalid_books, f, indent=2, ensure_ascii=False)

print("VALIDATION SUMMARY")
print("Valid records:", len(valid_books))
print("Invalid records:", len(invalid_books))
print("Unique books:", len(seen_urls))
print("Saved: output/books.json")
print("Saved: output/errors.json")

if valid_books:
    print("\nSAMPLE CLEAN RECORD:")
    print(json.dumps(valid_books[0], indent=2))


VALIDATION SUMMARY
Valid records: 60
Invalid records: 0
Unique books: 60
Saved: output/books.json
Saved: output/errors.json

SAMPLE CLEAN RECORD:
{
  "title": "A Light in the Attic",
  "product_url": "https://books.toscrape.com/catalogue/a-light-in-the-attic_1000/index.html",
  "price_text": "\u00c2\u00a351.77",
  "price_gbp": 51.77,
  "availability_text": "In stock (22 available)",
  "rating_text": "Three",
  "description": "It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Silverstein's humorous and creative verse can amuse the dowdiest of readers. Lemon-faced adults and fidgety kids sit still and read these rhythmic words and laugh and smile and love th It's hard to imagine a world without A Light in the Attic. This now-classic collection of poetry and drawings from Shel Silverstein celebrates its 20th anniversary with this special edition. Sil

In [ ]:

import time
import json
import requests
from datetime import datetime, timezone

test_started = time.time()

# Deliberately made-up book URL
fake_url = (
    "https://books.toscrape.com/catalogue/"
    "this-book-does-not-exist/index.html"
)

failed_pages = []

def test_broken_page(url):
    try:
        response = requests.get(
            url,
            headers=HEADERS,
            timeout=10
        )

        print("Test URL status:", response.status_code)

        if response.status_code == 200:
            return True

        if response.status_code in (403, 404):
            failed_pages.append({
                "url": url,
                "reason": f"HTTP {response.status_code}"
            })
            print("Page skipped. No retry.")
            return False

        if response.status_code >= 500:
            print("Server error. Retrying once...")
            time.sleep(1)

            retry = requests.get(
                url,
                headers=HEADERS,
                timeout=10
            )

            if retry.status_code != 200:
                failed_pages.append({
                    "url": url,
                    "reason": f"Retry returned {retry.status_code}"
                })
                return False

            return True

        failed_pages.append({
            "url": url,
            "reason": f"HTTP {response.status_code}"
        })
        return False

    except requests.Timeout:
        failed_pages.append({
            "url": url,
            "reason": "Request timed out"
        })
        return False

    except requests.RequestException as error:
        failed_pages.append({
            "url": url,
            "reason": str(error)
        })
        return False


# Test one deliberately broken page
test_broken_page(fake_url)

duration = round(time.time() - test_started, 2)

# Verify good records survived
with open("output/books.json", "r", encoding="utf-8") as f:
    saved_books = json.load(f)

report = {
    "started_at": datetime.now(timezone.utc).isoformat(),
    "duration_seconds": duration,
    "duration_scope": "Broken-page test only",
    "catalogue_pages": 3,
    "detail_pages_processed": 60,
    "pages_fetched": 63,
    "cache_hits": 2,
    "valid_records": len(saved_books),
    "invalid_records": len(invalid_books),
    "failed_pages": len(failed_pages),
    "failed_page_details": failed_pages
}

with open("output/run-report.json", "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2)

print("\nFAILURE HANDLING REPORT")
print("Valid records preserved:", len(saved_books))
print("Invalid records:", len(invalid_books))
print("Failed pages:", len(failed_pages))
print("Report saved: output/run-report.json")


Test URL status: 404
Page skipped. No retry.

FAILURE HANDLING REPORT
Valid records preserved: 60
Invalid records: 0
Failed pages: 1
Report saved: output/run-report.json


In [ ]:

import os
import json
import shutil

# Create the final project structure
os.makedirs("scraper/src", exist_ok=True)
os.makedirs("scraper/output", exist_ok=True)
os.makedirs("scraper/cache", exist_ok=True)

# Copy validated output files
for filename in [
    "books.json",
    "errors.json",
    "run-report.json"
]:
    source = os.path.join("output", filename)
    destination = os.path.join("scraper/output", filename)

    if os.path.exists(source):
        shutil.copy2(source, destination)
        print("Copied:", filename)
    else:
        print("Missing:", filename)

# Create requirements file
with open("scraper/requirements.txt", "w") as f:
    f.write(
        "requests\n"
        "beautifulsoup4\n"
        "pydantic\n"
    )

# Create gitignore
with open("scraper/.gitignore", "w") as f:
    f.write(
        "__pycache__/\n"
        "*.pyc\n"
        "cache/\n"
        ".env\n"
    )

print("\nPROJECT FOLDERS CREATED!")

print(os.listdir("scraper"))
print("Output files:", os.listdir("scraper/output"))


Copied: books.json
Copied: errors.json
Copied: run-report.json

PROJECT FOLDERS CREATED!
['output', 'src', 'cache', 'requirements.txt', '.gitignore']
Output files: ['run-report.json', 'errors.json', 'books.json']
